# Step 1: Data Loading
## Wing Shop Demand Forecasting Project

This notebook loads and merges datasets from Corporación Favorita (Kaggle) or generates sample data.

**Objectives:**
- Load raw data files (train, items, stores, oil, holidays)
- **Filter to only 1 store to reduce dataset size**
- Merge datasets into a single dataframe
- Generate sample data if Kaggle data is unavailable
- Save processed data for next steps

In [ ]:
# Import required libraries
import pandas as pd
import numpy as np
import os
import pickle
from datetime import datetime, timedelta
import warnings
warnings.filterwarnings('ignore')

# Configuration - Select which store to use (reduces dataset significantly)
SELECTED_STORE = 1  # Change this to use a different store (1-54 in Kaggle data)

print("Libraries imported successfully!")
print(f"\n*** FILTERING DATA TO STORE #{SELECTED_STORE} ONLY ***")

In [ ]:
# Setup project directories
BASE_DIR = os.path.dirname(os.getcwd()) if 'notebooks' in os.getcwd() else os.getcwd()
DATA_DIR = os.path.join(BASE_DIR, 'data')
MODELS_DIR = os.path.join(BASE_DIR, 'models')

# Create directories if they don't exist
os.makedirs(DATA_DIR, exist_ok=True)
os.makedirs(MODELS_DIR, exist_ok=True)

print(f"Base Directory: {BASE_DIR}")
print(f"Data Directory: {DATA_DIR}")
print(f"Models Directory: {MODELS_DIR}")

In [ ]:
# Utility functions
def load_csv(filename, **kwargs):
    """Load CSV file from data directory"""
    filepath = os.path.join(DATA_DIR, filename)
    if os.path.exists(filepath):
        return pd.read_csv(filepath, **kwargs)
    else:
        print(f"Warning: {filename} not found in {DATA_DIR}")
        return None

def save_model(model, name):
    """Save model to pickle file"""
    filepath = os.path.join(MODELS_DIR, f"{name}.pkl")
    with open(filepath, 'wb') as f:
        pickle.dump(model, f)
    print(f"Model saved: {filepath}")

def load_model(name):
    """Load model from pickle file"""
    filepath = os.path.join(MODELS_DIR, f"{name}.pkl")
    if os.path.exists(filepath):
        with open(filepath, 'rb') as f:
            return pickle.load(f)
    return None

def get_date_features(df, date_col='date'):
    """Extract date features for modeling"""
    df = df.copy()
    df[date_col] = pd.to_datetime(df[date_col])
    df['year'] = df[date_col].dt.year
    df['month'] = df[date_col].dt.month
    df['day'] = df[date_col].dt.day
    df['dayofweek'] = df[date_col].dt.dayofweek
    df['weekofyear'] = df[date_col].dt.isocalendar().week.astype(int)
    df['is_weekend'] = df['dayofweek'].isin([5, 6]).astype(int)
    df['is_month_start'] = df[date_col].dt.is_month_start.astype(int)
    df['is_month_end'] = df[date_col].dt.is_month_end.astype(int)
    return df

print("Utility functions defined!")

In [ ]:
# Generate sample data for ONLY 1 store
def generate_sample_data():
    """
    Generate sample data if Kaggle dataset is not available.
    Simulates realistic grocery store sales patterns for Cambodia market.
    Only generates data for 1 store to keep dataset small.
    """
    np.random.seed(42)
    
    # Generate dates for 2 years
    start_date = datetime(2024, 1, 1)
    dates = [start_date + timedelta(days=i) for i in range(730)]
    
    products = ['Rice', 'Water', 'Cooking Oil', 'Instant Noodles', 'Sugar', 'Soap', 'Milk', 'Bread']
    
    data = []
    store = SELECTED_STORE  # Only 1 store
    store_factor = np.random.uniform(0.8, 1.2)
    
    for product_idx, product in enumerate(products):
        # Base demand varies by product
        base_demand = [150, 200, 80, 120, 60, 40, 90, 100][product_idx]
        
        for date in dates:
            # Day of week effect (higher on weekends)
            dow_effect = 1.3 if date.weekday() >= 5 else 1.0
            
            # Monthly seasonality
            month_effect = 1 + 0.1 * np.sin(2 * np.pi * date.month / 12)
            
            # Holiday effect (simplified)
            is_holiday = date.day in [1, 15] or date.month == 12
            holiday_effect = 1.5 if is_holiday else 1.0
            
            # Economic shock (simulating Cambodia-Thailand conflict impact)
            if date >= datetime(2025, 6, 1):
                shock_effect = 0.85  # 15% reduction due to supply chain issues
            else:
                shock_effect = 1.0
            
            # Random promotion
            on_promotion = np.random.random() < 0.1
            promo_effect = 1.4 if on_promotion else 1.0
            
            # Calculate sales with noise
            sales = int(
                base_demand * store_factor * dow_effect * month_effect * 
                holiday_effect * shock_effect * promo_effect * 
                np.random.uniform(0.8, 1.2)
            )
            sales = max(0, sales)
            
            data.append({
                'date': date.strftime('%Y-%m-%d'),
                'store_nbr': store,
                'product': product,
                'family': ['GROCERY I', 'BEVERAGES', 'GROCERY I', 'GROCERY II', 
                          'GROCERY I', 'CLEANING', 'DAIRY', 'BREAD/BAKERY'][product_idx],
                'unit_sales': sales,
                'onpromotion': on_promotion,
                'is_holiday': is_holiday
            })
    
    df = pd.DataFrame(data)
    
    # Save generated data
    df.to_csv(os.path.join(DATA_DIR, 'sample_sales.csv'), index=False)
    print(f"Sample data generated: {len(df):,} records (1 store, 8 products, 730 days)")
    
    return df

print("Sample data generator defined (single store mode)!")

## 1.1 Load Kaggle Data (Single Store Only)

In [ ]:
print("=" * 60)
print("STEP 1: DATA LOADING (SINGLE STORE MODE)")
print("=" * 60)
print(f"\nLoading data for Store #{SELECTED_STORE} only...")

# Load only the selected store from train.csv to save memory
train_path = os.path.join(DATA_DIR, 'train.csv')

if os.path.exists(train_path):
    # Read in chunks and filter to single store
    print("\nReading train.csv in chunks (filtering to single store)...")
    chunks = []
    chunk_size = 500000
    
    for chunk in pd.read_csv(train_path, parse_dates=['date'], 
                              dtype={'onpromotion': str}, chunksize=chunk_size):
        filtered = chunk[chunk['store_nbr'] == SELECTED_STORE]
        chunks.append(filtered)
        print(f"  Processed chunk: kept {len(filtered):,} rows for store {SELECTED_STORE}")
    
    train = pd.concat(chunks, ignore_index=True)
    print(f"\nTotal rows after filtering: {len(train):,}")
else:
    train = None
    print("train.csv not found")

# Load other files (these are small)
items = load_csv('items.csv')
stores = load_csv('stores.csv')
oil = load_csv('oil.csv', parse_dates=['date'])
holidays = load_csv('holidays_events.csv', parse_dates=['date'])
transactions = load_csv('transactions.csv', parse_dates=['date'])

# Filter transactions to single store too
if transactions is not None:
    transactions = transactions[transactions['store_nbr'] == SELECTED_STORE]
    print(f"Transactions filtered: {len(transactions):,} rows")

In [ ]:
# Check if Kaggle data exists, otherwise generate sample data
if train is None:
    print("\nKaggle dataset not found. Generating sample data...")
    df = generate_sample_data()
else:
    print(f"\n[1/6] Train data (Store #{SELECTED_STORE}): {train.shape}")
    print(f"[2/6] Items data loaded: {items.shape if items is not None else 'N/A'}")
    print(f"[3/6] Stores data loaded: {stores.shape if stores is not None else 'N/A'}")
    print(f"[4/6] Oil data loaded: {oil.shape if oil is not None else 'N/A'}")
    print(f"[5/6] Holidays data loaded: {holidays.shape if holidays is not None else 'N/A'}")
    print(f"[6/6] Transactions (Store #{SELECTED_STORE}): {transactions.shape if transactions is not None else 'N/A'}")
    
    # Merge datasets
    print("\nMerging datasets...")
    
    # Merge items info
    if items is not None:
        df = train.merge(items, on='item_nbr', how='left')
    else:
        df = train.copy()
    
    # Merge store info
    if stores is not None:
        df = df.merge(stores, on='store_nbr', how='left')
    
    # Merge oil prices
    if oil is not None:
        df = df.merge(oil, on='date', how='left')
        df['dcoilwtico'] = df['dcoilwtico'].fillna(method='ffill')
    
    # Merge holidays
    if holidays is not None:
        holidays_national = holidays[holidays['locale'] == 'National'][['date', 'type', 'transferred']]
        holidays_national = holidays_national.rename(columns={'type': 'holiday_type'})
        df = df.merge(holidays_national, on='date', how='left')
        df['is_holiday'] = df['holiday_type'].notna().astype(int)
    
    # Merge transactions
    if transactions is not None:
        df = df.merge(transactions, on=['date', 'store_nbr'], how='left')
    
    print(f"\nFinal merged dataset (Store #{SELECTED_STORE} only): {df.shape}")

## 1.2 Aggregate Daily Sales

In [ ]:
def aggregate_daily_sales(df):
    """Aggregate sales by date, store, and product family."""
    print("\nAggregating daily sales by family...")
    
    # Check which columns exist
    group_cols = ['date']
    
    if 'store_nbr' in df.columns:
        group_cols.append('store_nbr')
    
    if 'family' in df.columns:
        group_cols.append('family')
    elif 'product' in df.columns:
        group_cols.append('product')
    
    # Aggregate
    agg_dict = {'unit_sales': 'sum'}
    
    if 'onpromotion' in df.columns:
        if df['onpromotion'].dtype == 'object':
            df['onpromotion'] = df['onpromotion'].map({'True': 1, 'False': 0, True: 1, False: 0}).fillna(0)
        agg_dict['onpromotion'] = 'sum'
    
    if 'transactions' in df.columns:
        agg_dict['transactions'] = 'mean'
    
    if 'dcoilwtico' in df.columns:
        agg_dict['dcoilwtico'] = 'mean'
    
    if 'is_holiday' in df.columns:
        agg_dict['is_holiday'] = 'max'
    
    df_agg = df.groupby(group_cols).agg(agg_dict).reset_index()
    
    print(f"Aggregated data shape: {df_agg.shape}")
    
    return df_agg

# Apply aggregation
df_agg = aggregate_daily_sales(df)

## 1.3 Add Date Features and Save

In [ ]:
# Add date features
df_agg = get_date_features(df_agg)

# Display data overview
print("\n" + "=" * 60)
print(f"DATA OVERVIEW (Store #{SELECTED_STORE} Only)")
print("=" * 60)
print(f"\nColumns: {list(df_agg.columns)}")
print(f"\nDate range: {df_agg['date'].min()} to {df_agg['date'].max()}")
print(f"\nData shape: {df_agg.shape}")

# Show unique families/products
if 'family' in df_agg.columns:
    print(f"\nUnique product families: {df_agg['family'].nunique()}")
    print(df_agg['family'].unique()[:10])
elif 'product' in df_agg.columns:
    print(f"\nUnique products: {df_agg['product'].nunique()}")
    print(df_agg['product'].unique())

In [ ]:
# Show sample data
df_agg.head(10)

In [ ]:
# Memory usage comparison
print("\n" + "=" * 60)
print("MEMORY SAVINGS")
print("=" * 60)
mem_usage = df_agg.memory_usage(deep=True).sum() / 1024 / 1024
print(f"Current dataset memory: {mem_usage:.2f} MB")
print(f"Estimated full dataset (54 stores): {mem_usage * 54:.2f} MB")
print(f"Memory saved by using 1 store: ~{mem_usage * 53:.2f} MB")

In [ ]:
# Save processed data
filepath = os.path.join(DATA_DIR, 'processed_sales.csv')
df_agg.to_csv(filepath, index=False)

print(f"\nProcessed data saved to: {filepath}")
print("\n" + "=" * 60)
print(f"DATA LOADING COMPLETE! (Store #{SELECTED_STORE} only)")
print("=" * 60)